**Fichier Python**



In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
#import matplotlib.pyplot as plt
#import seaborn as sns
from scipy import stats

**I. FICHIER MEMBERS**
---

In [ ]:
# ── 1. Open file ────────────────────────────────────────────
df_Members = pd.read_csv('./MEMBERS_DIM.csv', sep=None, engine="python")

In [ ]:
df_Members.head()

In [ ]:
# ── 2. Statistiques descriptives ────────────────────────────────────────────
print("\n2. Statistiques descriptives")
print("-" * 40)

colonnes_numeriques = df_Members.select_dtypes(include=[np.number]).columns.tolist()
# Exclude ID columns (high cardinality, not meaningful to describe)
colonnes_a_exclure = [c for c in colonnes_numeriques if "ID" in c.upper()]
colonnes_plot = [c for c in colonnes_numeriques if c not in colonnes_a_exclure]

desc = df_Members[colonnes_plot].describe().T  # keep for reference
print(desc)  # raw table still available if needed


In [ ]:
# ── 3. Spotting odd values ────────────────────────────────────────────
max_tenure = df_Members["TENURE_MONTHS"].max()
print(f"Le plus ancien adhérent existe depuis : {max_tenure} mois")
min_tenure = df_Members["TENURE_MONTHS"].min()
print(f"Le plus récent adhérent existe depuis : {min_tenure} mois")

min_age = df_Members["AGE"].min()
print(f"\nMin age : {min_age} ans")
max_age = df_Members["AGE"].max()
print(f"Max age : {max_age} ans")

nb_negatifs = (df_Members['CASH_BACK_POINTS_BALANCE'] < 0).sum()
print(f"\nNombre de lignes avec CASH_BACK_POINTS_BALANCE < 0 : {nb_negatifs}")
nb_negatifs = (df_Members['REWARD_POINTS_BALANCE'] < 0).sum()
print(f"Nombre de lignes avec REWARD_POINTS_BALANCE < 0 : {nb_negatifs}")

In [ ]:
# ── 4. Compter les valeurs manquantes et describe data type ────────────────────
print(f"\nNombre de rows dans le dataframe : ", len(df_Members))

# Compter les valeurs manquantes par colonne
print("\nNombre de valeurs manquantes par colonne :")
print(df_Members.isna().sum())
print("\nDatatypes :")
print(df_Members.dtypes)

In [ ]:
# ── 5. Gestion des valeurs manquantes ────────────────────────────────────────────


# 1. Remplacer les valeurs manquantes pour les colonnes de type object afin de conserver les observations pour la segmentation
df_Members_filled = df_Members.assign(
    EMAILABLE_FLAG=df_Members['EMAILABLE_FLAG'].fillna('Unknown'),
    LANGUAGE=df_Members['LANGUAGE'].fillna('Unknown'),
    PROV=df_Members['PROV'].fillna('Unknown'),
    AGE=df_Members['AGE'].fillna(0), 
    GENDER=df_Members['GENDER'].fillna('U')
)

""" Aucune des variables ci-dessus ne sera utilisée pour la segmentation, 
mais elles pourraient être utiles pour des analyses ultérieures ou pour enrichir les segments après la segmentation. 
En remplissant les valeurs manquantes avec des catégories spécifiques (comme 'Unknown' ou 0), 
nous conservons toutes les observations dans le dataset, ce qui est important pour la segmentation. """

# 2. Remplacer les valeurs négatives par 0 pour les données abérantes (âge négatif)
df_Members_filled['AGE'] = df_Members_filled['AGE'].apply(lambda x: 0 if x < 0 else x)


print(f"Nb Lignes avant nettoyage : {df_Members_filled.shape[0]}")
print(f"Nb Lignes après nettoyage : {df_Members_filled.shape[0]}")

In [ ]:
# ── 6. Retrait des valeurs supprenantes ────────────────────────────────────────────
''' 92 / 1254299  lignes avec des points de récompense négatifs, puisqu'il s'agit d'une immense minorité on les retire
#Si ses obs avaient été plus nombreuses, on aurait pu  faire un travail plus fin pour comprendre pourquoi elles sont négatives et les corriger au lieu de les supprimer
#mais pour le besoin de ce travail, cette tâche de nettoyage est suffisante '''

df_Members_pre_clean = df_Members_filled[
    (df_Members_filled['CASH_BACK_POINTS_BALANCE'] >= 0) &  # Garde les valeurs >= 0
    (df_Members_filled['REWARD_POINTS_BALANCE'] >= 0)  # Garde les valeurs >= 0
]

print(f"Nb Lignes avant nettoyage : {df_Members.shape[0]}")
print(f"Nb Lignes après nettoyage : {df_Members_pre_clean.shape[0]}")


In [ ]:
# ── 7. Def fonction outiler ────────────────────────────────────────────
def remove_outliers(df, column):
   mean = df[column].mean()
   std = df[column].std()
   df_clean = df[(df[column] >= mean - 2*std) & (df[column] <= mean + 2*std)]
   return df_clean
'''
Note : Cette étape est présentée à titre exploratoire. Dans un contexte réel, le retrait
des outliers serait une décision prise après une première analyse complète, seulement si
cela est justifié et susceptible d'améliorer les résultats du modèle K-Means.'''

print("\nRetrait des outliers : ")
print("-" * 40)

df_Memeber_clean = remove_outliers(df_Members_pre_clean, 'CASH_BACK_POINTS_BALANCE')
print(f"Nb Lignes avant nettoyage : {df_Members_pre_clean.shape[0]}")
print(f"Nb Lignes après nettoyage : {df_Memeber_clean.shape[0]}")

pourcentage_retiré = ((df_Members_pre_clean.shape[0] - df_Memeber_clean.shape[0]) / df_Members_pre_clean.shape[0]) * 100
print(f"\nPourcentage d'observations retirées : {pourcentage_retiré:.2f} %")

df_Member_final = remove_outliers(df_Memeber_clean, 'REWARD_POINTS_BALANCE')
print(f"\nNb Lignes avant nettoyage : {df_Memeber_clean.shape[0]}")
print(f"Nb Lignes après nettoyage : {df_Member_final.shape[0]}")

pourcentage_retiré = ((df_Memeber_clean.shape[0] - df_Member_final.shape[0]) / df_Memeber_clean.shape[0]) * 100
print(f"\nPourcentage d'observations retirées : {pourcentage_retiré:.2f} %")

#Verification du nombre de lignes après nettoyage et des valeurs manquantes
print("-" * 40)
print(f"\nNombre de rows dans le dataframe : ", len(df_Member_final))

# Compter les valeurs manquantes par colonne
print("\nNombre de valeurs manquantes par colonne :")
print(df_Members_pre_clean.isna().sum())
print("\nDatatypes :")
print(df_Members_pre_clean.dtypes)

In [ ]:
# ── 8. CSV Final ────────────────────────────────────────────
# Drop TIER, colonnes indiquée comme inutile par le client 
df_Member_final = df_Member_final.drop(columns=['TIER'])
df_Member_final.head(50)

# Exporter le dataframe nettoyé
df_Member_final.to_csv("df_Members_final.csv", index=False)

**II.FICHIER REWARD**
---

Ce fichier contient deux années du programme de fidelité Big_Smile

In [ ]:
# ── 1. Open file ────────────────────────────────────────────
df_Rewards = pd.read_csv('./REWARD_FACT.csv', sep=None, engine="python")
df_Rewards.head()

In [ ]:
# ── 2. Description ────────────────────────────────────────────
print("\n2. Statistiques descriptives")
print("-" * 40)
# Sélectionner uniquement les colonnes numériques
colonnes_numeriques = df_Rewards.select_dtypes(include=[np.number]).columns.tolist()
colonnes_a_exclure = [c for c in colonnes_numeriques if "ID" in c.upper()] # Exclude ID columns 
colonnes_plot = [c for c in colonnes_numeriques if c not in colonnes_a_exclure]

desc = df_Rewards[colonnes_plot].describe().T  
print(desc) 

In [ ]:
# ── 3. Valeurs manquantes ────────────────────────────────────────────
# # Compter les valeurs manquantes par colonne
print("\nNombre de valeurs manquantes par colonne :")
print(df_Rewards.isna().sum())

In [ ]:
# ── 4. Comprendre REWARDS_CATEGORY  ────────────────────────────────────────────
num_unique_values =df_Rewards["REWARDS_CATEGORY"].nunique()
print(f"Nombre de valeurs uniques dans REWARDS_CATEGORY : {num_unique_values}")

print("\nDistribution des catégories de récompenses :")
df_Rewards["REWARDS_CATEGORY"].value_counts()

## Création de dummies variables en vue de Kmeans

In [ ]:
# ── 5. Création de dummies POINTS_REDEEMED / REWARDS_CATEGORY  ────────────────────────────
# Colonnes pour lesquelles créer des dummies multipliées par REWARDS_CATEGORY
cols_to_multiply = ['POINTS_REDEEMED', 'NUMBER_ITEMS_REDEEMED', 'REDEMPTIONS']

df_result = df_Rewards.copy()

for col in cols_to_multiply:
    df_hot = pd.get_dummies(df_Rewards['REWARDS_CATEGORY'], prefix=f'{col}_PER', drop_first=False)
    
    for dummy_col in df_hot.columns:
        df_hot[dummy_col] = df_hot[dummy_col] * df_Rewards[col]
    
    df_result = pd.concat([df_result, df_hot], axis=1)


In [ ]:
pd.set_option('display.max_columns', None)
df_result.head(5)

In [ ]:
#supprimer la colonne Reward_category maintenant que nous avons les dummiers
df_result.drop(columns=['REWARDS_CATEGORY'], inplace=True)

In [ ]:
# ── 6. Rassembler le dataset par MEMBER_ID  (une ligne par membre)  ──────────────

# Création du dictionnaire d'agrégation pour conserver le dernier mois de Redepmptions:
agg_dict = {col: 'sum' for col in df_result.columns if col not in ['MEMBER_ID', 'MONTH_ID']}
agg_dict['MONTH_ID'] = 'max'

# Application de l'agrégation par MEMBER_ID
df_Rewards_aggregated = df_result.groupby('MEMBER_ID').agg(agg_dict).reset_index()

df_Rewards_aggregated.head(5)

**Création de variables**
---

In [3]:
# ── 7. Calcul des ratios d'utilisation des points par catégorie
'''Afin de condenser le jeu de données et d'avoir des variables plus parlantes pour la segmentation, 
on va calculer, et ne conserver uniquement, que les ratios d'utilisation des points par catégorie.
Exemple : Nb points utilisés dans chaque catégorie / total de points utilisés (POINTS_REDEEMED) '''

# Liste des catégories de points à transformer en pourcentage
categories_r = [
    "POINTS_REDEEMED_PER_Cash_Back",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_1",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_2",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_3",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_4"
]

# Création des nouvelles variables en pourcentage
for cat in categories_r:
    df_Rewards_aggregated[f'%_{cat}'] = (df_Rewards_aggregated[cat] / df_Rewards_aggregated['POINTS_REDEEMED']) * 100

categories_p = [
    "REDEMPTIONS_PER_Cash_Back",
    "REDEMPTIONS_PER_REWARD_CATEGORY_1",
    "REDEMPTIONS_PER_REWARD_CATEGORY_2",
    "REDEMPTIONS_PER_REWARD_CATEGORY_3",
    "REDEMPTIONS_PER_REWARD_CATEGORY_4"
]

# Création des nouvelles variables en pourcentage
for cat in categories_p:
    df_Rewards_aggregated[f'%_{cat}'] = (df_Rewards_aggregated[cat] / df_Rewards_aggregated['REDEMPTIONS']) * 100

categories_n = [
    "NUMBER_ITEMS_REDEEMED_PER_Cash_Back",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_1",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_2",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_3",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_4"
]

# Création des nouvelles variables en pourcentage
for cat in categories_n:
    df_Rewards_aggregated[f'%_{cat}'] = (df_Rewards_aggregated[cat] / df_Rewards_aggregated['NUMBER_ITEMS_REDEEMED']) * 100
    
# Vérification
df_Rewards_aggregated.head()

NameError: name 'df_Rewards_aggregated' is not defined

In [ ]:
#Suppression des variables redondantes

cols_to_drop = [
    "POINTS_REDEEMED_PER_Cash_Back",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_1",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_2",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_3",
    "POINTS_REDEEMED_PER_REWARD_CATEGORY_4",
    "REDEMPTIONS_PER_Cash_Back",
    "REDEMPTIONS_PER_REWARD_CATEGORY_1",
    "REDEMPTIONS_PER_REWARD_CATEGORY_2",
    "REDEMPTIONS_PER_REWARD_CATEGORY_3",
    "REDEMPTIONS_PER_REWARD_CATEGORY_4",
    "NUMBER_ITEMS_REDEEMED_PER_Cash_Back",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_1",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_2",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_3",
    "NUMBER_ITEMS_REDEEMED_PER_REWARD_CATEGORY_4"
]

df_Rewards_aggregated = df_Rewards_aggregated.drop(columns=cols_to_drop)

# Vérification
df_Rewards_aggregated.head()

In [ ]:
# ── 8. Valeur de récence de la dernière fois qu'un membre à REDEEMED une récompense

# #Création d'une varibale de récence de la dernière fois qu'un membre à REDEEMED une récompense
#Pour cette étape, je pars du postulat que nous somme en février 20XX et que le membre n'a pas commandé ce  mois-là
def months_diff(last_order, reference):
    """
    Calcule la différence en mois entre last_order et reference.
    Les deux paramètres sont au format AAAAMM (ex: 202411 pour novembre 2024).
    """
    last_year = last_order // 100       # Extrait l'année (ex: 202411 // 100 = 2024)
    last_month = last_order % 100        # Extrait le mois (ex: 202411 % 100 = 11)
    ref_year = reference // 100          # Extrait l'année de référence
    ref_month = reference % 100          # Extrait le mois de référence
    
    return (ref_year - last_year) * 12 + (ref_month - last_month)

reference_month = 202502  # Par exemple, février 2025

df_Rewards_aggregated['MONTHS_SIN_LAST_REDEMP'] = df_Rewards_aggregated['MONTH_ID'].apply(
    lambda x: months_diff(x, reference_month))

print("\nCette opération a permis de créer une nouvelle variable MONTHS_SIN_LAST_REDEMP \nqui indique le nombre de mois écoulés depuis la dernière fois qu'un membre a REDEEMED une récompense.")
# Affichage des 5 premières lignes pour vérifier
df_Rewards_aggregated.head(5)


**SCORE DE FIDELITÉ**


In [ ]:
# ── 8. SCORE DE FIDELITÉ RFM
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Créer une colonne pour l'inverse de MONTHS_SIN_LAST_REDEMP  type : (1/(x+1) pour éviter la division par zéro)
'''Un membre qui a racheté récemment a peu de mois écoulés → petite valeur → après inversion, grande valeur
Le +1 évite une division par zéro si le membre a racheté ce mois-ci
Résultat : plus tu es récent, plus ton score est élevé, '''

df_Rewards_aggregated['MONTHS_SIN_LAST_REDEMP_INV'] = 1 / (df_Rewards_aggregated['MONTHS_SIN_LAST_REDEMP'] + 1)

# 2. Sélectionner les variables à standardiser
# On utilise la variable inversée pour la récence ainsi que POINTS_REDEEMED et NUMBER_ITEMS_REDEEMED
cols_to_scale = ['MONTHS_SIN_LAST_REDEMP_INV', 'POINTS_REDEEMED', 'NUMBER_ITEMS_REDEEMED']

# On applique le MinMaxScaler pour redimensionner ces variables entre 0 et 1 et ne pas avoir de score négatif
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_values = scaler.fit_transform(df_Rewards_aggregated[cols_to_scale])

# On transforme le résultat en DataFrame 
scaled_df = pd.DataFrame(scaled_values, 
                         columns=[col + '_scaled' for col in cols_to_scale],
                         index=df_Rewards_aggregated.index)

# Concaténer les colonnes standardisées avec le DataFrame d'origine
df_Rewards_aggregated = pd.concat([df_Rewards_aggregated, scaled_df], axis=1)

# Calculer un score composite, par exemple en utilisant les coefficients souhaités
#    Ici, on utilise 0.2 pour la récence inversée, 0.4 pour POINTS_REDEEMED, et 0.6 pour NUMBER_ITEMS_REDEEMED.
#    On utilisera les versions standardisées pour que toutes les variables soient sur la même échelle.
df_Rewards_aggregated['SCORE_BS_USAGE'] = (
    df_Rewards_aggregated['MONTHS_SIN_LAST_REDEMP_INV_scaled'] * 0.2 +
    df_Rewards_aggregated['POINTS_REDEEMED_scaled'] * 0.4 +
    df_Rewards_aggregated['NUMBER_ITEMS_REDEEMED_scaled'] * 0.6
)
# Multiplier le score par 1000 afin d'agrandir l'échelle et d'améliorer la lisibilité
df_Rewards_aggregated['SCORE_BS_USAGE'] = df_Rewards_aggregated['SCORE_BS_USAGE'] * 1000

NameError: name 'df_Rewards_aggregated' is not defined

In [ ]:
# Renommer MONTH_ID pour la cohérence
df_Rewards_aggregated = df_Rewards_aggregated.rename(columns={'MONTH_ID': 'LAST_REW'})
df_Rewards_aggregated.head(5)

In [ ]:
# ── 9. Labelling RFM basé sur une logique client pyramidale
'''pd.qcut() : Cette fonction découpe une variable continue en intervalles basés sur des quantiles.
#q=[0, 0.6, 0.9, 1.0] :
#De 0 à 0.6 (soit les 60 % inférieurs) seront classés en Bronze.
#De 0.6 à 0.9 (les 30 % suivants) seront classés en Silver.
#De 0.9 à 1.0 (les 10 % supérieurs) seront classés en Gold.'''

df_Rewards_aggregated['FIDELITY_LABEL'] = pd.qcut(
    df_Rewards_aggregated['SCORE_BS_USAGE'], 
    q=[0, 0.6, 0.9, 1.0], 
    labels=['Bronze', 'Silver', 'Gold'])

In [ ]:
# ── 10. CSV Final pour l'utlisation des points
cols_to_drop = ['MONTHS_SIN_LAST_REDEMP_INV', 'MONTHS_SIN_LAST_REDEMP_INV_scaled', 'POINTS_REDEEMED_scaled', 'NUMBER_ITEMS_REDEEMED_scaled']
df_Rewards_aggregated = df_Rewards_aggregated.drop(columns=cols_to_drop)

df_Rewards_aggregated.to_csv("df_Rewards_final.csv", index=False)


**III. FICHIER TRANSACTIONS**
---
  

Un troisième fichier de données a été fourni par le client, 
mais afin de garder ce portfolio concis, le code ne sera pas présenté ici.

A third data file was provided by the client, but in order to keep this portfolio concise, the code will not be presented here.

